In [2]:
import os

# Replace these with your actual paths
os.environ["PATH"] += r";C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin"
os.environ["XTBPATH"] = r"C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\share\xtb"

# Check it's found
import shutil
print("xtb location:", shutil.which("xtb"))


xtb location: C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE


In [13]:
def clean_orca_restart_files():
    for f in os.listdir('.'):
        if f.endswith(('.gbw', '.chk', '.rst', '.prop', '.tmp')):
            os.remove(f)


In [37]:
# Step 2: Comprehensive Feature Extraction with RDKit, Mordred, and xTB
import os
import re
import subprocess
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors, AllChem
from mordred import Calculator, descriptors
import re

failed = []

# --- Helpers for 3D geometry and file I/O ---
def generate_3d_structure(smiles: str, xyz_path: str):
    """Embed in 3D, UFF-optimize, and write an XYZ file."""
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.UFFOptimizeMolecule(mol)
    conf = mol.GetConformer()
    with open(xyz_path, 'w') as f:
        f.write(f"{mol.GetNumAtoms()}\n\n")
        for atom in mol.GetAtoms():
            x, y, z = conf.GetAtomPosition(atom.GetIdx())
            f.write(f"{atom.GetSymbol()} {x:.6f} {y:.6f} {z:.6f}\n")
    print(f"  • Wrote 3D structure to {xyz_path}")

def run_xtb_dft(
    xyz_path: str,
    log_path: str,
    n_cores: int = 4,
    xtb_executable: str = None
) -> bool:
    """
    Call xTB to compute GFN2-xTB properties.
    Returns True on success, False if xTB fails.
    """
    # Locate xTB binary
    xtb_cmd = xtb_executable or shutil.which("xtb")
    if xtb_cmd is None:
        raise FileNotFoundError(
            "xTB executable not found. Please install xTB or set `xtb_executable`."
        )

    cmd = [
        xtb_cmd,
        xyz_path,
        "--gfn2",
        "--parallel", str(n_cores),
        "--verbose"
    ]

    print(f"    • Running xTB via `{xtb_cmd}` on {os.path.basename(xyz_path)}")
    with open(log_path, "w") as logf:
        try:
            subprocess.run(cmd, stdout=logf, stderr=subprocess.STDOUT, check=True)
            print(f"    • Logged xTB output to {log_path}")
            return True
        except subprocess.CalledProcessError as e:
            print(f"    ⚠ xTB failed (exit code {e.returncode}) on {xyz_path}, skipping DFT features")
            return False


def parse_xtb_output(log_path: str) -> dict:
    """Pull out HOMO-LUMO gap, dipole moment, and polarizability from xTB log."""
    props = {}
    # open with utf-8 and ignore bad bytes
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()

    # HOMO-LUMO gap [eV]
    m = re.search(r"HOMO-LUMO GAP\s+\[eV\]\s+([\d\.\-]+)", text)
    if m:
        props['HOMO_LUMO_gap_eV'] = float(m.group(1))

    # Dipole moment [Debye]
    m = re.search(r"DIPOLE MOMENT.*?total\s+([\d\.\-]+)", text, re.DOTALL)
    if m:
        props['Dipole_moment_Debye'] = float(m.group(1))

    # Polarizability [au]
    m = re.search(r"POLARIZABILITY\s+\[au\]\s+([\d\.\-]+)", text)
    if m:
        props['Polarizability_au'] = float(m.group(1))

    print(f"    → Parsed xTB props: {props}")
    return props


# --- Descriptor calculators ---
mordred_calc = Calculator(descriptors, ignore_3D=True)

def compute_rdkit_descriptors(smiles: str) -> dict:
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        return {}
    desc = {
        'MolWt_RDKit': Descriptors.MolWt(mol),
        'LogP_RDKit': Crippen.MolLogP(mol),
        'TPSA_RDKit': rdMolDescriptors.CalcTPSA(mol),
        'HBD': rdMolDescriptors.CalcNumHBD(mol),
        'HBA': rdMolDescriptors.CalcNumHBA(mol),
        'RotatableBonds': Descriptors.NumRotatableBonds(mol),
        'NumRings': mol.GetRingInfo().NumRings()
    }
    return desc

# --- Main pipeline step for feature extraction ---
def extract_step2_features(
    input_csv: str,
    output_csv: str,
    temp_dir: str = "dft_tmp",
    n_cores: int = 4
):
    os.makedirs(temp_dir, exist_ok=True)
    df = pd.read_csv(input_csv)
    records = []

    for idx, row in df.iterrows():
        smiles = row['Additive_SMILES']
        print(f"\nProcessing row {idx} — SMILES: {smiles}")

        # RDKit + Mordred as before...
        rdkit_feats = compute_rdkit_descriptors(smiles)
        m_desc = mordred_calc(Chem.MolFromSmiles(smiles))
        mordred_series = pd.Series(m_desc.asdict()).dropna()

        # Generate geometry
        xyz_file = os.path.join(temp_dir, f"mol_{idx}.xyz")
        log_file = os.path.join(temp_dir, f"mol_{idx}_xtb.log")
        generate_3d_structure(smiles, xyz_file)

        # Run xTB, catch failures
        success = run_xtb_dft(xyz_file, log_file, n_cores=n_cores)
        if success:
            xtb_feats = parse_xtb_output(log_file)
        else:
            xtb_feats = {}
            failed.append((idx, smiles))

        # Merge and append
        merged = {**row.to_dict(), **rdkit_feats, **mordred_series.to_dict(), **xtb_feats}
        records.append(merged)

    out_df = pd.DataFrame(records)
    out_df.to_csv(output_csv, index=False)
    print(f"\n✅ Step 2 complete — enriched features saved to {output_csv}")
    print(f"⚠️ Failed xTB runs: {len(failed)}")
    print(failed)


# Example usage:
extract_step2_features("s1.csv", "enriched_features.csv", n_cores=8)




Processing row 0 — SMILES: OCC(F)(F)F
  • Wrote 3D structure to dft_tmp\mol_0.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_0.xyz
    • Logged xTB output to dft_tmp\mol_0_xtb.log
    → Parsed xTB props: {}

Processing row 1 — SMILES: CC1COC(=O)O1
  • Wrote 3D structure to dft_tmp\mol_1.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_1.xyz
    • Logged xTB output to dft_tmp\mol_1_xtb.log
    → Parsed xTB props: {}

Processing row 2 — SMILES: CC#N
  • Wrote 3D structure to dft_tmp\mol_2.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_2.xyz
    • Logged xTB output to dft_tmp\mol_2_xtb.log
    → Parsed xTB props: {}

Processing row 3 — SMILES: OCC(F)(F)F
  • Wrote 3D structure to dft_tmp\mol_3.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_3.

c:\DataFiles\Projects\SummerResearch4\venv310\lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


  • Wrote 3D structure to dft_tmp\mol_129.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_129.xyz
    • Logged xTB output to dft_tmp\mol_129_xtb.log
    → Parsed xTB props: {}

Processing row 130 — SMILES: C(C(=O)O)N
  • Wrote 3D structure to dft_tmp\mol_130.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_130.xyz
    • Logged xTB output to dft_tmp\mol_130_xtb.log
    → Parsed xTB props: {}

Processing row 131 — SMILES: COC(=O)[C@H](CO)N
  • Wrote 3D structure to dft_tmp\mol_131.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_131.xyz
    • Logged xTB output to dft_tmp\mol_131_xtb.log
    → Parsed xTB props: {}

Processing row 132 — SMILES: C1=CC=C2C(=C1)C(=O)C3=CC(=C(C(=C3C2=O)O)O)S(=O)(=O)[O-].[Na+]
  • Wrote 3D structure to dft_tmp\mol_132.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pr

[12:06:31] UFFTYPER: Unrecognized charge state for atom: 2
[12:06:31] UFFTYPER: Unrecognized charge state for atom: 4


  • Wrote 3D structure to dft_tmp\mol_155.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_155.xyz
    ⚠ xTB failed (exit code 128) on dft_tmp\mol_155.xyz, skipping DFT features

Processing row 156 — SMILES: [Na+].[Na+].[Na+].[Na+].[O-][P]([O-])(=O)O[P]([O-])([O-])=O
  • Wrote 3D structure to dft_tmp\mol_156.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_156.xyz
    ⚠ xTB failed (exit code 128) on dft_tmp\mol_156.xyz, skipping DFT features

Processing row 157 — SMILES: [Na+].[K+].O[C@H]([C@@H](O)C([O-])=O)C([O-])=O
  • Wrote 3D structure to dft_tmp\mol_157.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_157.xyz
    ⚠ xTB failed (exit code 128) on dft_tmp\mol_157.xyz, skipping DFT features

Processing row 158 — SMILES: C1=CC2=C(C=C1S(=O)(=O)[O-])C(=O)C3=C(C2=O)C=CC(=C3)S(=O)(=O)[O-].[Na+].[Na+]
  • Wrote

...

    ⚠ xTB failed (exit code 128) on dft_tmp\mol_173.xyz, skipping DFT features

Processing row 174 — SMILES: [Na+].[Na+].[Na+].[Na+].[O-][P]([O-])(=O)O[P]([O-])([O-])=O
  • Wrote 3D structure to dft_tmp\mol_174.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_174.xyz
    ⚠ xTB failed (exit code 128) on dft_tmp\mol_174.xyz, skipping DFT features

Processing row 175 — SMILES: [Na+].[K+].O[C@H]([C@@H](O)C([O-])=O)C([O-])=O
  • Wrote 3D structure to dft_tmp\mol_175.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_175.xyz
    ⚠ xTB failed (exit code 128) on dft_tmp\mol_175.xyz, skipping DFT features

Processing row 176 — SMILES: [Na+].[O-][Cl](=O)(=O)=O
  • Wrote 3D structure to dft_tmp\mol_176.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_176.xyz
    • Logged xTB output to dft_tmp\mol_176_xtb.log
    → Pars

[12:06:37] UFFTYPER: Warning: hybridization set to SP3 for atom 1
[12:06:37] UFFTYPER: Unrecognized charge state for atom: 1
[12:06:37] UFFTYPER: Warning: hybridization set to SP3 for atom 2
[12:06:37] UFFTYPER: Unrecognized charge state for atom: 2


    • Logged xTB output to dft_tmp\mol_205_xtb.log
    → Parsed xTB props: {}

Processing row 206 — SMILES: C([C@H]([C@H]([C@@H]([C@H](C(=O)[O-])O)O)O)O)O.[Na+]
  • Wrote 3D structure to dft_tmp\mol_206.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_206.xyz
    • Logged xTB output to dft_tmp\mol_206_xtb.log
    → Parsed xTB props: {}

Processing row 207 — SMILES: [Na+].[Na+].[Na+].OC(CC([O-])=O)(CC([O-])=O)C([O-])=O
  • Wrote 3D structure to dft_tmp\mol_207.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_207.xyz
    ⚠ xTB failed (exit code 128) on dft_tmp\mol_207.xyz, skipping DFT features

Processing row 208 — SMILES: CC1=CC=C(C=C1)S(=O)(=O)[O-].[Na+]
  • Wrote 3D structure to dft_tmp\mol_208.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_208.xyz
    • Logged xTB output to dft_tmp\mol_208_xtb.log
   

[12:06:38] UFFTYPER: Warning: hybridization set to SP3 for atom 1
[12:06:38] UFFTYPER: Unrecognized charge state for atom: 1
[12:06:38] UFFTYPER: Warning: hybridization set to SP3 for atom 2
[12:06:38] UFFTYPER: Unrecognized charge state for atom: 2


  • Wrote 3D structure to dft_tmp\mol_212.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_212.xyz
    • Logged xTB output to dft_tmp\mol_212_xtb.log
    → Parsed xTB props: {}

Processing row 213 — SMILES: [Na+].[Na+].[Na+].OC(CC([O-])=O)(CC([O-])=O)C([O-])=O
  • Wrote 3D structure to dft_tmp\mol_213.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_213.xyz
    ⚠ xTB failed (exit code 128) on dft_tmp\mol_213.xyz, skipping DFT features

Processing row 214 — SMILES: [Na+].[O-]C(=O)c1ccccc1
  • Wrote 3D structure to dft_tmp\mol_214.xyz
    • Running xTB via `C:\Users\nkapa\Downloads\xtb-6.7.1pre-windows-x86_64\xtb-6.7.1\bin\xtb.EXE` on mol_214.xyz
    • Logged xTB output to dft_tmp\mol_214_xtb.log
    → Parsed xTB props: {}

Processing row 215 — SMILES: C(F)(F)(F)S(=O)(=O)[O-].[Na+]
  • Wrote 3D structure to dft_tmp\mol_215.xyz
    • Running xTB via `C:\Users\nkapa\Dow

In [22]:
import os
import re
import shutil
import subprocess
import warnings

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem

# suppress noisy warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# point to xtb binary (autodetect if available in PATH)
XTB_CMD = shutil.which("xtb")

# ──────────────────────────────────────────────────────────────────────────────
def remove_counterions(smiles: str) -> str:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return smiles
    frags = Chem.GetMolFrags(mol, asMols=True)
    keep = [
        f for f in frags
        if all(ion not in Chem.MolToSmiles(f) for ion in ("[Na+]", "[K+]", "[I-]"))
    ]
    if not keep:
        return smiles
    main = max(keep, key=lambda m: m.GetNumAtoms())
    return Chem.MolToSmiles(main)

def get_net_charge(smiles: str) -> int:
    mol = Chem.MolFromSmiles(smiles)
    return sum(atom.GetFormalCharge() for atom in mol.GetAtoms()) if mol else 0

def generate_3d(smiles: str, xyz_path: str) -> None:
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.UFFOptimizeMolecule(mol)
    conf = mol.GetConformer()
    with open(xyz_path, "w") as f:
        f.write(f"{mol.GetNumAtoms()}\n\n")
        for atom in mol.GetAtoms():
            x, y, z = conf.GetAtomPosition(atom.GetIdx())
            f.write(f"{atom.GetSymbol()} {x:.6f} {y:.6f} {z:.6f}\n")

def run_xtb(xyz_path: str, log_path: str, n_cores: int = 4, charge: int = None) -> bool:
    if XTB_CMD is None:
        raise RuntimeError("xtb not found in PATH.")
    cmd = [XTB_CMD, xyz_path, "--gfn2", "--parallel", str(n_cores), "--verbose"]
    if charge is not None:
        cmd += ["--chrg", str(charge)]
    with open(log_path, "w") as logf:
        try:
            subprocess.run(cmd, stdout=logf, stderr=subprocess.STDOUT, check=True)
            return True
        except subprocess.CalledProcessError:
            return False

def parse_xtb_log(log_path: str) -> dict:
    props = {}
    with open(log_path, "r", errors="ignore") as fh:
        text = fh.read()

    patterns = {
        "total_energy_Eh": r"TOTAL ENERGY\s+([-\d\.]+)\s+Eh",
        "gradient_norm_Eh_a0": r"GRADIENT NORM\s+([-\d\.]+)\s+Eh",
        "homo_lumo_gap_eV": r"HOMO-LUMO GAP\s+([-\d\.]+)\s+eV",
        "homo_eig_eV": r"HOMO orbital eigv\.\s+([-\d\.]+)\s+eV",
        "lumo_eig_eV": r"LUMO orbital eigv\.\s+([-\d\.]+)\s+eV",
        "atomisation_energy_Eh": r"atomisation energy\s+([-\d\.]+)\s+Eh",
    }

    for key, pat in patterns.items():
        m = re.search(pat, text)
        if m:
            props[key] = float(m.group(1))

    m = re.search(
        r"molecular dipole:[\s\S]*?full:\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)",
        text,
    )
    if m:
        props.update({
            "dipole_x_D": float(m.group(1)),
            "dipole_y_D": float(m.group(2)),
            "dipole_z_D": float(m.group(3)),
            "dipole_tot_D": float(m.group(4)),
        })

    return props

def process_smiles_df(df: pd.DataFrame, tmp_folder: str = "dft_tmp", cores: int = 4) -> pd.DataFrame:
    os.makedirs(tmp_folder, exist_ok=True)
    records = []
    failures = []

    for idx, row in df.iterrows():
        smi = row["Additive_SMILES"]
        print(f"\nProcessing row {idx}: {smi}")

        xyz_path = os.path.join(tmp_folder, f"{idx}.xyz")
        log_path = os.path.join(tmp_folder, f"{idx}.log")

        generate_3d(smi, xyz_path)
        success = run_xtb(xyz_path, log_path, cores)

        if not success:
            stripped = remove_counterions(smi)
            charge = get_net_charge(stripped)
            if stripped != smi:
                print(f"  → retrying with cleaned SMILES: {stripped}")
                generate_3d(stripped, xyz_path)
                success = run_xtb(xyz_path, log_path, cores, charge)

        xtb_data = parse_xtb_log(log_path) if success else {}
        if not success:
            print("  ⚠ Failed even after retry")
            failures.append((idx, smi))

        rec = row.to_dict()
        rec.update(xtb_data)
        records.append(rec)

    out_df = pd.DataFrame.from_records(records)
    out_df = out_df.loc[:, ~out_df.columns.duplicated()]
    
    if failures:
        print(f"\n{len(failures)} molecules failed. Example: {failures[:1]}")
    else:
        print("\nAll molecules processed successfully.")

    return out_df

df = pd.read_csv("s1.csv")  # your input file with Additive_SMILES
result = process_smiles_df(df, tmp_folder="dft_tmp", cores=8)
result.to_csv("xtb_output.csv", index=False)



Processing row 0: OCC(F)(F)F

Processing row 1: CC1COC(=O)O1

Processing row 2: CC#N

Processing row 3: OCC(F)(F)F

Processing row 4: C1CCOC1

Processing row 5: CC1COC(=O)O1

Processing row 6: CN(C)C(C)=O

Processing row 7: OCC(F)(F)F

Processing row 8: C1CCOC1

Processing row 9: CC1COC(=O)O1

Processing row 10: CN(C)C(C)=O

Processing row 11: CC#N

Processing row 12: C(C(F)(F)F)(C(F)(F)F)O

Processing row 13: CCOC(=O)C(C)O

Processing row 14: [C@H]12CCC([C@H](OC1)O2)=O

Processing row 15: COCCOC

Processing row 16: CCOC(=O)C(C)O

Processing row 17: CN(C)C(=O)C(F)(F)F

Processing row 18: COCCOC

Processing row 19: OCC(O)CO

Processing row 20: COCCOC

Processing row 21: CS(=O)C

Processing row 22: CO[C@H]1COC2[C@@H](COC12)OC

Processing row 23: C(CN(CC(=O)O)CC(=O)[O-])N(CC(=O)O)CC(=O)[O-].[Na+].[Na+]
  → retrying with cleaned SMILES: O=C([O-])CN(CCN(CC(=O)[O-])CC(=O)O)CC(=O)O

Processing row 24: C1=CC=C(C=C1)NC2=CC=C(C=C2)S(=O)(=O)[O-].[Na+]

Processing row 25: CS(=O)C

Processing row 

[11:31:52] UFFTYPER: Unrecognized charge state for atom: 2
[11:31:52] UFFTYPER: Unrecognized charge state for atom: 4
[11:31:52] UFFTYPER: Unrecognized charge state for atom: 4
[11:31:52] UFFTYPER: Unrecognized charge state for atom: 4



Processing row 157: [Na+].[K+].O[C@H]([C@@H](O)C([O-])=O)C([O-])=O
  → retrying with cleaned SMILES: O=C([O-])[C@H](O)[C@@H](O)C(=O)[O-]

Processing row 158: C1=CC2=C(C=C1S(=O)(=O)[O-])C(=O)C3=C(C2=O)C=CC(=C3)S(=O)(=O)[O-].[Na+].[Na+]
  → retrying with cleaned SMILES: O=C1c2ccc(S(=O)(=O)[O-])cc2C(=O)c2cc(S(=O)(=O)[O-])ccc21

Processing row 159: FC(F)(F)S(=O)(=O)N([Na])S(=O)(=O)C(F)(F)F

Processing row 160: [Li+].C(F)(F)(F)S(=O)(=O)[N-]S(=O)(=O)C(F)(F)F

Processing row 161: [Na+].[Na+].[O-][S]([S-])(=O)=O
  → retrying with cleaned SMILES: O=S(=O)([O-])[S-]


[11:31:53] UFFTYPER: Unrecognized charge state for atom: 2
[11:31:53] UFFTYPER: Unrecognized charge state for atom: 4
[11:31:53] UFFTYPER: Unrecognized charge state for atom: 4
[11:31:53] UFFTYPER: Unrecognized charge state for atom: 4



Processing row 162: [Na+].[Na+].[Na+].[Na+].[O-][P]([O-])(=O)O[P]([O-])([O-])=O
  → retrying with cleaned SMILES: O=P([O-])([O-])OP(=O)([O-])[O-]

Processing row 163: [Na+].[K+].O[C@H]([C@@H](O)C([O-])=O)C([O-])=O
  → retrying with cleaned SMILES: O=C([O-])[C@H](O)[C@@H](O)C(=O)[O-]

Processing row 164: C1=CC2=C(C=C1S(=O)(=O)[O-])C(=O)C3=C(C2=O)C=CC(=C3)S(=O)(=O)[O-].[Na+].[Na+]
  → retrying with cleaned SMILES: O=C1c2ccc(S(=O)(=O)[O-])cc2C(=O)c2cc(S(=O)(=O)[O-])ccc21

Processing row 165: FC(F)(F)S(=O)(=O)N([Na])S(=O)(=O)C(F)(F)F

Processing row 166: [Li+].C(F)(F)(F)S(=O)(=O)[N-]S(=O)(=O)C(F)(F)F

Processing row 167: [Na+].[Na+].[O-][S]([S-])(=O)=O
  → retrying with cleaned SMILES: O=S(=O)([O-])[S-]


...


Processing row 174: [Na+].[Na+].[Na+].[Na+].[O-][P]([O-])(=O)O[P]([O-])([O-])=O
  → retrying with cleaned SMILES: O=P([O-])([O-])OP(=O)([O-])[O-]

Processing row 175: [Na+].[K+].O[C@H]([C@@H](O)C([O-])=O)C([O-])=O
  → retrying with cleaned SMILES: O=C([O-])[C@H](O)[C@@H](O)C(=O)[O-]

Processing row 176: [Na+].[O-][Cl](=O)(=O)=O

Processing row 177: C1=CC2=C(C=C1S(=O)(=O)[O-])C(=O)C3=C(C2=O)C=CC(=C3)S(=O)(=O)[O-].[Na+].[Na+]
  → retrying with cleaned SMILES: O=C1c2ccc(S(=O)(=O)[O-])cc2C(=O)c2cc(S(=O)(=O)[O-])ccc21

Processing row 178: FC(F)(F)S(=O)(=O)N([Na])S(=O)(=O)C(F)(F)F

Processing row 179: [Li+].C(F)(F)(F)S(=O)(=O)[N-]S(=O)(=O)C(F)(F)F

Processing row 180: COC(=O)[C@@H](CO)N

Processing row 181: COC(=O)[C@H](CO)N

Processing row 182: C([C@H](C(=O)O)N)C(=O)N

Processing row 183: N1C=CC=C1

Processing row 184: COC(=O)[C@@H](CO)N

Processing row 185: COC(=O)[C@H](CO)N

Processing row 186: C(C(C(C(=O)[O-])O)C(=O)[O-])C(=O)[O-].[Na+].[Na+].[Na+]
  → retrying with cleaned SMILES: O=C(

[11:31:59] UFFTYPER: Warning: hybridization set to SP3 for atom 1
[11:31:59] UFFTYPER: Unrecognized charge state for atom: 1
[11:31:59] UFFTYPER: Warning: hybridization set to SP3 for atom 2
[11:31:59] UFFTYPER: Unrecognized charge state for atom: 2



Processing row 206: C([C@H]([C@H]([C@@H]([C@H](C(=O)[O-])O)O)O)O)O.[Na+]

Processing row 207: [Na+].[Na+].[Na+].OC(CC([O-])=O)(CC([O-])=O)C([O-])=O
  → retrying with cleaned SMILES: O=C([O-])CC(O)(CC(=O)[O-])C(=O)[O-]

Processing row 208: CC1=CC=C(C=C1)S(=O)(=O)[O-].[Na+]

Processing row 209: CC(=O)[O-].[Na+]

Processing row 210: [Na+].F[P-](F)(F)(F)(F)F

Processing row 211: [Na+].[Na+].[O-][S]([O-])(=O)=O
  → retrying with cleaned SMILES: O=S(=O)([O-])[O-]


[11:32:00] UFFTYPER: Warning: hybridization set to SP3 for atom 1
[11:32:00] UFFTYPER: Unrecognized charge state for atom: 1
[11:32:00] UFFTYPER: Warning: hybridization set to SP3 for atom 2
[11:32:00] UFFTYPER: Unrecognized charge state for atom: 2



Processing row 212: C([C@H]([C@H]([C@@H]([C@H](C(=O)[O-])O)O)O)O)O.[Na+]

Processing row 213: [Na+].[Na+].[Na+].OC(CC([O-])=O)(CC([O-])=O)C([O-])=O
  → retrying with cleaned SMILES: O=C([O-])CC(O)(CC(=O)[O-])C(=O)[O-]

Processing row 214: [Na+].[O-]C(=O)c1ccccc1

Processing row 215: C(F)(F)(F)S(=O)(=O)[O-].[Na+]

Processing row 216: [Na+].[Na+].[O-][S]([O-])(=O)=O
  → retrying with cleaned SMILES: O=S(=O)([O-])[O-]

Processing row 217: C([C@H]([C@H]([C@@H]([C@H](C(=O)[O-])O)O)O)O)O.[Na+]

Processing row 218: [Na+].[Na+].[Na+].OC(CC([O-])=O)(CC([O-])=O)C([O-])=O
  → retrying with cleaned SMILES: O=C([O-])CC(O)(CC(=O)[O-])C(=O)[O-]

Processing row 219: [Na+].[O-]C(=O)c1ccccc1

Processing row 220: CC(=O)[O-].[Na+]

Processing row 221: C(F)(F)(F)S(=O)(=O)[O-].[Na+]

Processing row 222: [Na+].[Na+].[O-][S]([O-])(=O)=O
  → retrying with cleaned SMILES: O=S(=O)([O-])[O-]

Processing row 223: C1=CC=C2C(=C1)N=C3C=CC=CC3=N2

Processing row 224: C1=CC=C2C(=C1)NC3=CC=CC=C3S2

Processing row 225: 

In [32]:
import os, re, csv
from typing import Dict, List

# ---------------------------------------------------------------------------
# Regex patterns to grab numbers from the xTB log
LOG_PATTERNS = {
    "total_energy_Eh":       r"TOTAL ENERGY\s+([-\d\.]+)\s+Eh",
    "gradient_norm_Eh_a0":   r"GRADIENT NORM\s+([-\d\.]+)\s+Eh",
    "homo_lumo_gap_eV":      r"HOMO-LUMO GAP\s+([-\d\.]+)\s+eV",
    "homo_eig_eV":           r"HOMO orbital eigv\.\s+([-\d\.]+)\s+eV",
    "lumo_eig_eV":           r"LUMO orbital eigv\.\s+([-\d\.]+)\s+eV",
}

DIPOLE_REGEX = re.compile(
    r"molecular dipole:[\s\S]*?full:\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)",
    re.MULTILINE,
)

# ---------------------------------------------------------------------------
def parse_log(path: str, debug: bool = False) -> Dict[str, float]:
    props: Dict[str, float] = {}
    
    if not os.path.isfile(path):
        if debug:
            print(f"[DEBUG]   ❌ LOG not found: {path}")
        return props
    
    with open(path, "r", errors="ignore") as fh:
        text = fh.read()
    
    if debug:
        print(f"[DEBUG]   ✅ Read {len(text):,} characters from {os.path.basename(path)}")
    
    # pattern matches
    for key, pat in LOG_PATTERNS.items():
        m = re.search(pat, text)
        if m:
            props[key] = float(m.group(1))
            if debug:
                print(f"          • matched {key}: {props[key]}")
        elif debug:
            print(f"          • NO match for {key}")
    
    # dipole block
    m = DIPOLE_REGEX.search(text)
    if m:
        props.update({
            "dipole_x_debye":   float(m.group(1)),
            "dipole_y_debye":   float(m.group(2)),
            "dipole_z_debye":   float(m.group(3)),
            "dipole_total_debye": float(m.group(4)),
        })
        if debug:
            print(f"          • matched dipole components {props['dipole_total_debye']} D")
    elif debug:
        print("          • NO dipole match")
    
    return props


def parse_xyz(path: str, debug: bool = False) -> Dict[str, str]:
    if not os.path.isfile(path):
        if debug:
            print(f"[DEBUG]   ❌ XYZ not found: {path}")
        return {}
    
    with open(path) as fh:
        lines = [ln.rstrip() for ln in fh]
    
    try:
        num_atoms = int(lines[0])
    except Exception:
        num_atoms = None
    
    if debug:
        print(f"[DEBUG]   ✅ Read {len(lines)} XYZ lines; atoms = {num_atoms}")
    
    coords_block = "\n".join(lines[2:])  # skip header + comment
    return {"num_atoms": num_atoms, "xyz_block": coords_block}


def collect_data(folder: str, start: int, end: int, debug_every: int = 1) -> List[Dict[str, object]]:
    rows: List[Dict[str, object]] = []
    
    for idx in range(start, end + 1):
        debug = (idx - start) % debug_every == 0  # print every Nth molecule
        if debug:
            print(f"\n[MOL {idx}]  -----------------------------")
        
        row = {"id": idx}
        row.update(parse_log(os.path.join(folder, f"{idx}.log"), debug))
        row.update(parse_xyz(os.path.join(folder, f"{idx}.xyz"), debug))
        rows.append(row)
    
    return rows


def write_csv(rows: List[Dict[str, object]], out_file: str) -> None:
    all_keys = {"id"}
    for r in rows:
        all_keys.update(r)
    fieldnames = ["id"] + sorted(k for k in all_keys if k != "id")
    
    with open(out_file, "w", newline="") as fh:
        csv.DictWriter(fh, fieldnames).writeheader()
        csv.DictWriter(fh, fieldnames).writerows(rows)

# ---------------------------------------------------------------------------
# ‼️  RUN THIS SECTION ‼️
folder = "dft_tmp"   # where 0.log, 0.xyz, 1.log, ... live
rows   = collect_data(folder, start=0, end=571, debug_every=1)  # change debug_every if it’s too noisy
print("\n[SUMMARY] Extracted keys:", set().union(*(r.keys() for r in rows)))
write_csv(rows, "xtb_debug_out.csv")



[MOL 0]  -----------------------------
[DEBUG]   ✅ Read 17,950 characters from 0.log
          • matched total_energy_Eh: -24.063838453541
          • matched gradient_norm_Eh_a0: 0.072309877691
          • matched homo_lumo_gap_eV: 11.170661192755
          • matched homo_eig_eV: -12.636004418348
          • matched lumo_eig_eV: -1.465343225593
          • matched dipole components 2.867 D
[DEBUG]   ✅ Read 11 XYZ lines; atoms = 9

[MOL 1]  -----------------------------
[DEBUG]   ✅ Read 18,859 characters from 1.log
          • matched total_energy_Eh: -23.845771208809
          • matched gradient_norm_Eh_a0: 0.140616806028
          • matched homo_lumo_gap_eV: 5.302371882217
          • matched homo_eig_eV: -12.143832082838
          • matched lumo_eig_eV: -6.841460200621
          • matched dipole components 6.38 D
[DEBUG]   ✅ Read 15 XYZ lines; atoms = 13

[MOL 2]  -----------------------------
[DEBUG]   ✅ Read 16,955 characters from 2.log
          • matched total_energy_Eh: -8.686

In [33]:
# --- ML-ready feature builder for xTB output -------------------------------
import pandas as pd
import numpy as np
from itertools import combinations
from collections import Counter
from rdkit.Chem import rdchem, PeriodicTable

# ←———— edit here if file lives elsewhere
RAW_CSV = "xtb_debug_out.csv"
OUT_CSV = "xtb_ml_ready.csv"

# ───────────────────────── helpers ──────────────────────────
ptable = rdchem.GetPeriodicTable()

# covalent radii in Å (Pyykkö 2009, a lite subset)
COV_RAD = {
    1: 0.31,  6: 0.76, 7: 0.71, 8: 0.66, 9: 0.57,
    14: 1.11, 15: 1.07, 16: 1.05, 17: 1.02,
}
def elem_to_Z(elem): 
    return ptable.GetAtomicNumber(elem)
    

def parse_xyz_block(block: str):
    """Return (elements[list[str]], coords[np.ndarray shape (N,3)])"""
    elems, xyz = [], []
    for line in block.strip().splitlines():
        if not line.strip():           # skip blank lines
            continue
        parts = line.split()
        elems.append(parts[0])
        xyz.append([float(x) for x in parts[1:4]])
    return elems, np.array(xyz)

def bonds_from_coords(elems, coords, scale=1.2):
    """Return list of (i,j,dist_Å) of putative covalent bonds."""
    bonds = []
    for i, j in combinations(range(len(elems)), 2):
        Zi, Zj = elem_to_Z(elems[i]), elem_to_Z(elems[j])
        ri = COV_RAD.get(Zi, 0.77)   # fall-back radius
        rj = COV_RAD.get(Zj, 0.77)
        cutoff = scale * (ri + rj)
        d = np.linalg.norm(coords[i] - coords[j])
        if d <= cutoff:
            bonds.append((i, j, d))
    return bonds

def bond_angles(bonds, coords):
    """Return list of bond angles (degrees) for connected triples."""
    # map each atom to neighbours
    nbrs = {}
    for i, j, _ in bonds:
        nbrs.setdefault(i, []).append(j)
        nbrs.setdefault(j, []).append(i)
    angs = []
    for center, neighs in nbrs.items():
        for a, b in combinations(neighs, 2):
            v1 = coords[a] - coords[center]
            v2 = coords[b] - coords[center]
            cosang = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
            angs.append(np.degrees(np.arccos(np.clip(cosang, -1.0, 1.0))))
    return angs

def moment_of_inertia(coords, masses):
    """Return eigenvalues (I1 ≤ I2 ≤ I3) of moment-of-inertia tensor."""
    com = np.average(coords, axis=0, weights=masses)
    xyz = coords - com
    I = np.zeros((3, 3))
    for m, (x, y, z) in zip(masses, xyz):
        I += m * np.array([
            [y*y+z*z, -x*y,    -x*z   ],
            [-x*y,    x*x+z*z, -y*z   ],
            [-x*z,    -y*z,    x*x+y*y],
        ])
    evals = np.linalg.eigvalsh(I)
    return np.sort(evals)            # ascending

def add_geometry_features(row):
    elems, coords = parse_xyz_block(row["xyz_block"])
    masses = np.array([ptable.GetAtomicWeight(e) for e in elems])

    # -------------- bond-based
    bond_list = bonds_from_coords(elems, coords)
    blens = [d for *_, d in bond_list]
    b_angs = bond_angles(bond_list, coords)

    # -------------- size / shape
    com = np.average(coords, axis=0, weights=masses)
    rg = np.sqrt(np.average(np.sum((coords-com)**2, axis=1), weights=masses))
    I1, I2, I3 = moment_of_inertia(coords, masses)

    # -------------- element counts
    elem_counts = Counter(elems)
    elem_feat = {f"n_{el}": elem_counts.get(el, 0) for el in elem_counts}

    # -------------- assemble dict
    feat = {
        # bond stats
        "n_bonds": len(blens),
        "bond_len_mean": np.mean(blens) if blens else np.nan,
        "bond_len_std":  np.std(blens)  if blens else np.nan,
        "bond_ang_mean": np.mean(b_angs) if b_angs else np.nan,
        "bond_ang_std":  np.std(b_angs)  if b_angs else np.nan,
        # size/shape
        "radius_gyration": rg,
        "I1": I1, "I2": I2, "I3": I3,
        # element makeup
        **elem_feat,
    }
    # dipole magnitude recompute as a sanity check
    feat["dipole_calc_D"] = np.linalg.norm([row["dipole_x_debye"],
                                            row["dipole_y_debye"],
                                            row["dipole_z_debye"]])
    # energy per atom
    feat["energy_per_atom"] = row["total_energy_Eh"] / row["num_atoms"]
    # gap in nm
    feat["HL_gap_nm"] = 1239.841984 / row["homo_lumo_gap_eV"] if row["homo_lumo_gap_eV"] else np.nan
    return feat

# ───────────────────────── main build ──────────────────────────
raw = pd.read_csv(RAW_CSV)

# drop obviously unusable text block after feature extraction
geom_feats = raw.apply(add_geometry_features, axis=1, result_type="expand")
df_ml = pd.concat([raw.drop(columns=["xyz_block"]),
                   geom_feats], axis=1)

# finally, ensure purely numeric and handle NaNs
numeric_cols = df_ml.columns.difference(["id"])
df_ml[numeric_cols] = df_ml[numeric_cols].apply(pd.to_numeric, errors="coerce")
df_ml = df_ml.fillna(df_ml.mean())        # simple imputation; tweak as needed

df_ml.to_csv(OUT_CSV, index=False)
print(f"✅  ML-ready file written → {OUT_CSV}")


✅  ML-ready file written → xtb_ml_ready.csv


In [36]:
import pandas as pd

# Load both files
df_xtb = pd.read_csv("xtb_ml_ready.csv")
df_other = pd.read_csv("output.csv")

# Drop 'id' from xtb_ml_ready if present
df_xtb = df_xtb.drop(columns=["id"], errors="ignore")

# Check that number of rows match
if len(df_xtb) != len(df_other):
    raise ValueError(f"Row count mismatch: xtb_ml_ready.csv has {len(df_xtb)} rows, "
                     f"but output.csv has {len(df_other)} rows.")

# Merge horizontally (side by side)
df_combined = pd.concat([df_other, df_xtb], axis=1)

# Save to output
df_combined.to_csv("final_features.csv", index=False)

print(f"✅ Merged horizontally: final_features.csv with {df_combined.shape[0]} rows and {df_combined.shape[1]} columns")


✅ Merged horizontally: final_features.csv with 572 rows and 1554 columns


In [ ]:
# Step 3: Robust Feature Selection Pipeline
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
import shap
from boruta import BorutaPy
from sklearn.feature_selection import RFE
#import ace_tools as tools  # for interactive display

# 1. Load data
df = pd.read_csv("enriched_features.csv")

# 2. Define target and features
target = "CE_aver. (%)"
exclude = ['#','Additive_SMILES','IUPAC_NAME','PubChemCID','ChEMBL_ID',
           'CE_1 (%)','CE_2 (%)','CE_3 (%)', target]
features = [c for c in df.columns if c not in exclude]

# 3. Prepare X and y
X = df[features].fillna(df[features].mean())
y = df[target]

# 4. Spearman correlation ranking
spearman_scores = {
    feat: abs(spearmanr(X[feat], y)[0])
    for feat in features
}
spearman_rank = pd.Series(spearman_scores).sort_values(ascending=False)
top_spearman = spearman_rank.head(50).index.tolist()

# 5. SHAP importance ranking
rf_model = RandomForestRegressor(n_estimators=100, random_state=0)
rf_model.fit(X, y)
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X)
shap_importance = np.mean(np.abs(shap_values), axis=0)
shap_rank = pd.Series(shap_importance, index=features).sort_values(ascending=False)
top_shap = shap_rank.head(50).index.tolist()

# 6. Boruta selection
boruta = BorutaPy(
    estimator=RandomForestRegressor(n_estimators=100, random_state=0),
    n_estimators='auto', verbose=0, random_state=0
)
boruta.fit(X.values, y.values)
boruta_selected = [features[i] for i, keep in enumerate(boruta.support_) if keep]

# 7. RFE selection
rfe = RFE(
    estimator=RandomForestRegressor(n_estimators=100, random_state=0),
    n_features_to_select=50, step=10
)
rfe.fit(X, y)
rfe_selected = [f for f, sel in zip(features, rfe.support_) if sel]

# 8. Aggregate selections
results = pd.DataFrame({
    'feature': features,
    'spearman': [spearman_scores[f] for f in features],
    'shap': [shap_rank[f] for f in features],
    'boruta': [1 if f in boruta_selected else 0 for f in features],
    'rfe':    [1 if f in rfe_selected else 0 for f in features],
    'in_top_spearman': [1 if f in top_spearman else 0 for f in features],
    'in_top_shap':     [1 if f in top_shap else 0 for f in features],
})
# Count how many methods selected each feature
methods = ['boruta','rfe','in_top_spearman','in_top_shap']
results['score'] = results[methods].sum(axis=1)

# 9. Define "viable features" as those selected by at least two methods
viable = results.query("score >= 2").sort_values(by='score', ascending=False)

# 10. Display the viable feature list
#tools.display_dataframe_to_user(
#    name="Viable Features for CE Prediction",
#    dataframe=viable[['feature','score']]
#)

from IPython.display import display
display(dataframe=viable[['feature','score']])



C:\Users\nkapa\AppData\Local\Temp\ipykernel_16224\739310291.py:12: DtypeWarning: Columns (154,155,163,164,181,182,190,191,199,200,208,209,217,218,226,227,235,236,244,245,250,251,252,253,254,255,256,257,258,358,359,360,361,362,370,371,379,380,397,398,406,407,415,416,424,425,433,434,442,443,451,452,460,461,466,467,468,469,476,477,482,483,484,485,500,501,508,509,516,517,524,525,532,533,540,541,548,549,556,557,562,563,564,565,572,573,578,579,580,581,596,597,604,605,612,613,620,621,628,629,636,637,644,645,652,653,797,798,808,844,860,1317) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("enriched_features.csv")
C:\Users\nkapa\AppData\Local\Temp\ipykernel_16224\739310291.py:33: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  feat: abs(spearmanr(X[feat], y)[0])


KeyboardInterrupt: 

In [6]:
import matplotlib.pyplot as plt

# 10a. Display as table
print("Viable Features (score >= 2):")
display(viable[['feature','score']].reset_index(drop=True))

# 10b. Plot top 20 by score
top20 = viable.head(20)
plt.figure(figsize=(8,6))
plt.barh(top20['feature'], top20['score'])
plt.gca().invert_yaxis()
plt.xlabel('Selection Score')
plt.title('Top 20 Viable Features')
plt.tight_layout()
plt.show()

Viable Features (score >= 2):


NameError: name 'viable' is not defined

In [ ]:
# ── After your Step 3 cell, where you have `viable` ──

# 1) Pull the feature names:
features = viable['feature'].tolist()
print(f"Using {len(features)} viable features for modeling.")

# 2) Load your full dataset:
df = pd.read_csv("enriched_features.csv")
X = df[features].copy().fillna(df[features].mean())
y = df["CE_aver. (%)"].copy()

# 3) Then paste in the Step 4 code, but use these X, y:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, make_scorer
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import joblib
from scipy.stats import randint, uniform

# Train/test split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Hyperparameter search space
param_dist = {
    "n_estimators": randint(100, 500),
    "max_depth": randint(3, 12),
    "learning_rate": uniform(0.01, 0.2),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "gamma": uniform(0, 5),
    "reg_lambda": uniform(0, 5),
}

xgb_reg = xgb.XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    tree_method="gpu_hist",       # use the GPU‐accelerated tree builder
    predictor="gpu_predictor",    # use GPU for predictions, too
    gpu_id=0,                     # index of your GPU (0 for the first one)
)

search = RandomizedSearchCV(
    xgb_reg,
    param_distributions=param_dist,
    n_iter=30,
    scoring=make_scorer(r2_score),
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

print("[INFO] Starting hyperparameter tuning...")
search.fit(X_train, y_train)

best_model = search.best_estimator_
print("[RESULT] Best params:", search.best_params_)

# Evaluate on validation set
preds = best_model.predict(X_val)
print(f"R²   = {r2_score(y_val, preds):.3f}")
print(f"MAE  = {mean_absolute_error(y_val, preds):.3f}")
print(f"RMSE = {mean_squared_error(y_val, preds, squared=False):.3f}")

# SHAP summary plot
explainer = shap.TreeExplainer(best_model)
shap_vals = explainer.shap_values(X_val)
plt.figure(figsize=(8,6))
shap.summary_plot(shap_vals, X_val, show=False)
plt.title("SHAP Summary — XGBoost CE Model")
plt.tight_layout()
plt.savefig("shap_summary_xgb.png")
plt.close()

# Save model
joblib.dump(best_model, "xgb_best_model.pkl")
print("[INFO] Model saved to xgb_best_model.pkl")


NameError: name 'viable' is not defined